In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Représenter les ordinateurs quantiques pour le transpileur


<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Pour convertir un circuit abstrait en circuit ISA pouvant s'exécuter sur un QPU (unité de traitement quantique) spécifique, le transpileur a besoin de certaines informations sur ce QPU. Ces informations se trouvent à deux endroits : l'objet `BackendV2` (ou l'ancien `BackendV1`) vers lequel tu prévois de soumettre des jobs, et l'attribut `Target` du backend.

- Le [`Target`](../api/qiskit/qiskit.transpiler.Target) contient toutes les contraintes pertinentes d'un appareil, telles que ses portes de base natives, la connectivité des qubits, et les informations d'impulsion ou de timing.
- Le [`Backend`](../api/qiskit/qiskit.providers.BackendV2) possède un `Target` par défaut, contient des informations supplémentaires — comme l'[`InstructionScheduleMap`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.pulse.InstructionScheduleMap) — et fournit l'interface pour soumettre des jobs de circuit quantique.

Tu peux également fournir explicitement des informations au transpileur, par exemple si tu as un cas d'usage spécifique, ou si tu penses que ces informations l'aideront à générer un circuit plus optimisé.

La précision avec laquelle le transpileur produit le circuit le plus adapté à un matériel spécifique dépend de la quantité d'informations sur ses contraintes que contient le `Target` ou le `Backend`.

> **Note:** Comme beaucoup des algorithmes de transpilation sous-jacents sont stochastiques, il n'est pas garanti qu'un meilleur circuit sera trouvé.

Cette page présente plusieurs exemples de transmission d'informations QPU au transpileur. Ces exemples utilisent la cible du backend fictif [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

<span id="default-config"></span>
## Configuration par défaut
La façon la plus simple d'utiliser le transpileur est de lui fournir toutes les informations sur le QPU via le `Backend` ou le `Target`. Pour mieux comprendre comment fonctionne le transpileur, construis un circuit et transpile-le avec différentes informations, comme suit.

Importe les bibliothèques nécessaires et instancie le QPU :

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
target = backend.target

L'exemple de circuit utilise une instance d'[`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) de la bibliothèque de circuits de Qiskit.

In [2]:
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)

qc.draw("mpl")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg)

Cet exemple utilise les paramètres par défaut pour transpiler vers le `target` du `backend`, qui fournit toutes les informations nécessaires pour convertir le circuit en un circuit pouvant s'exécuter sur le backend.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=target, seed_transpiler=12345
)
qc_t_target = pass_manager.run(qc)
qc_t_target.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg)

Cet exemple est utilisé dans les sections suivantes de ce sujet pour illustrer que la carte de couplage et les portes de base sont les informations essentielles à transmettre au transpileur pour une construction optimale du circuit. Le QPU peut généralement sélectionner des paramètres par défaut pour les autres informations non transmises, comme le timing et la planification.

## Carte de couplage

La carte de couplage est un graphe qui montre quels qubits sont connectés et donc disposent de portes à deux qubits entre eux. Ce graphe est parfois directionnel, ce qui signifie que les portes à deux qubits ne peuvent aller que dans un seul sens. Cependant, le transpileur peut toujours inverser la direction d'une porte en ajoutant des portes à qubit unique supplémentaires. Un circuit quantique abstrait peut toujours être représenté sur ce graphe, même si sa connectivité est limitée, en introduisant des portes SWAP pour déplacer l'information quantique.

Les qubits de nos circuits abstraits sont appelés _qubits virtuels_ et ceux de la carte de couplage sont des _qubits physiques_. Le transpileur fournit une correspondance entre les qubits virtuels et les qubits physiques. L'une des premières étapes de la transpilation, l'étape de _layout_, effectue cette correspondance.

> **Note:** Bien que l'étape de routage soit étroitement liée à l'étape de _layout_ — qui sélectionne les qubits réels — par défaut, ce sujet les traite comme des étapes séparées pour simplifier. La combinaison du routage et du layout est appelée _mapping de qubits_. Pour en savoir plus sur ces étapes, consulte le sujet [Étapes du transpileur](transpiler-stages).

Passe l'argument nommé `coupling_map` pour voir son effet sur le transpileur :

In [4]:
coupling_map = target.build_coupling_map()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv0 = pass_manager.run(qc)
qc_t_cm_lv0.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg)

Comme indiqué ci-dessus, plusieurs portes SWAP ont été insérées (chacune composée de trois portes CX), ce qui entraînera beaucoup d'erreurs sur les appareils actuels. Pour voir quels qubits sont sélectionnés sur la topologie réelle des qubits, utilise `plot_circuit_layout` des visualisations Qiskit :

In [5]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv0, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg)

Cela montre que nos qubits virtuels 0 à 11 ont été trivialement mappés à la ligne de qubits physiques 0 à 11. Revenons au niveau par défaut (`optimization_level=1`), qui utilise `VF2Layout` si un routage est nécessaire.

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv1 = pass_manager.run(qc)
qc_t_cm_lv1.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg)

À présent, aucune porte SWAP n'est insérée et les qubits physiques sélectionnés sont les mêmes qu'avec la classe `target`.

In [7]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv1, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg)

Maintenant le layout est en anneau. Comme ce layout respecte la connectivité du circuit, il n'y a aucune porte SWAP, ce qui donne un circuit bien meilleur pour l'exécution.

## Portes de base

Chaque ordinateur quantique prend en charge un ensemble d'instructions limité, appelé ses _portes de base_. Chaque porte du circuit doit être traduite en éléments de cet ensemble. Cet ensemble doit être composé de portes à un et deux qubits qui forment un ensemble de portes universel, ce qui signifie que toute opération quantique peut être décomposée en ces portes. Cette opération est effectuée par le [BasisTranslator](../api/qiskit/qiskit.transpiler.passes.BasisTranslator), et les portes de base peuvent être spécifiées comme argument nommé du transpileur pour fournir cette information.

In [8]:
basis_gates = list(target.operation_names)
print(basis_gates)

['sx', 'switch_case', 'x', 'if_else', 'measure', 'for_loop', 'delay', 'ecr', 'id', 'reset', 'rz']


The default single-qubit gates on _ibm_sherbrooke_ are `rz`, `x`, and `sx`, and the default two-qubit gate is `ecr` (echoed cross-resonance). CX gates are constructed from `ecr` gates, so on some QPUs `ecr` is specified as the two-qubit basis gate, while on others `cx` is the default. The `ecr` gate is the _entangling_ part of the `cx` gate. In addition to the control gates, there are also `delay` and `measurement` instructions.


<Admonition type="note">
    QPUs have default basis gates, but you can choose whatever gates you want, as long as you provide the instruction or add pulse gates (see [Create transpiler passes.](custom-transpiler-pass)) The default basis gates are those that calibrations have been done for on the QPU, so no further instruction/pulse gates need to be provided. For example, on some QPUs `cx` is the default two-qubit gate and `ecr` on others. See the list of possible [native gates and operations](/docs/guides/qpu-information#native-gates) for more details.
</Admonition>

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    coupling_map=coupling_map,
    basis_gates=basis_gates,
    seed_transpiler=12345,
)
qc_t_cm_bg = pass_manager.run(qc)
qc_t_cm_bg.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg" alt="Output of the previous code cell" />

Les portes à qubit unique par défaut sur _ibm_sherbrooke_ sont `rz`, `x` et `sx`, et la porte à deux qubits par défaut est `ecr` (echoed cross-resonance). Les portes CX sont construites à partir de portes `ecr`, donc sur certains QPUs, `ecr` est spécifiée comme porte de base à deux qubits, tandis que sur d'autres, `cx` est la valeur par défaut. La porte `ecr` est la partie _d'intrication_ de la porte `cx`. En plus des portes de contrôle, il y a aussi des instructions `delay` et `measurement`.

> **Note:** Les QPUs ont des portes de base par défaut, mais tu peux choisir les portes que tu veux, du moment que tu fournis l'instruction ou que tu ajoutes des portes d'impulsion (voir [Créer des passes de transpilation.](custom-transpiler-pass)). Les portes de base par défaut sont celles pour lesquelles des calibrations ont été effectuées sur le QPU, donc aucune instruction supplémentaire ni porte d'impulsion n'est nécessaire. Par exemple, sur certains QPUs `cx` est la porte à deux qubits par défaut et `ecr` sur d'autres. Consulte la liste des [portes et opérations natives possibles](/guides/qpu-information#native-gates) pour plus de détails.

In [10]:
target["ecr"][(1, 0)]

InstructionProperties(duration=5.333333333333332e-07, error=0.007494257741828603)

![Sortie de la cellule de code précédente](../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg)

Note que les objets `CXGate` ont été décomposés en portes `ecr` et en portes de base à qubit unique.
## Taux d'erreur de l'appareil
La classe `Target` peut contenir des informations sur les taux d'erreur des opérations sur l'appareil.
Par exemple, le code suivant récupère les propriétés de la porte echoed cross-resonance (ECR) entre le qubit 1 et le qubit 0 (note que la porte ECR est directionnelle) :

In [11]:
from qiskit.transpiler import Target
from qiskit.circuit.controlflow import IfElseOp, SwitchCaseOp, ForLoopOp

err_targ = Target.from_configuration(
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    num_qubits=target.num_qubits,
    custom_name_mapping={
        "if_else": IfElseOp,
        "switch_case": SwitchCaseOp,
        "for_loop": ForLoopOp,
    },
)

for i, (op, qargs) in enumerate(target.instructions):
    if op.name in basis_gates:
        err_targ[op.name][qargs] = target.instruction_properties(i)

Transpile with our new target `err_targ` as the target:

In [12]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=err_targ, seed_transpiler=12345
)
qc_t_cm_bg_et = pass_manager.run(qc)
qc_t_cm_bg_et.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/f1e270c4-e2cc-487e-a050-4180bc321b0b-0.svg" alt="Output of the previous code cell" />

La sortie affiche la durée de la porte (en secondes) et son taux d'erreur. Pour révéler les informations d'erreur au transpileur, construis un modèle de cible avec les `basis_gates` et la `coupling_map` ci-dessus et remplis-le avec les valeurs d'erreur du backend `FakeSherbrooke`.